# 06 — Fluid Bed Coating Digital Twin — Validation

Runs the full process chain **Pre-heating → Spraying → Drying** with parameters
read directly from `data/DoE_recipe_and_inputs.csv`, then overlays model predictions
against measured process data and dissolution profiles for any of the 20 DoE runs.

**Select a run (1–20) with the slider** — all data and plots update automatically.

* **Temperature panel**: inlet setpoint (dashed) · model outlet & product (lines) · measured outlet & product (+ markers)
* **Acetone / WG panels**: model predictions only
* **Dissolution panel**: model curves (lines) and measured aliquot averages (+ markers) for both sampling times (end-of-spray and discharge)


In [22]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from fluid_bed.parameters          import ProcessParameters
from fluid_bed.models.preheating   import run_preheating
from fluid_bed.models.spraying     import run_spraying
from fluid_bed.models.drying_stage import run_drying, calc_r_drying

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

# Hardware / physical constants
_RHO_AIR  = 1.10    # kg/m3
_CP_AIR   = 1010.0  # J/kg/K
_RHO_PART = 1050.0  # kg/m3
_CP_PART  = 1400.0  # J/kg/K
_D_BED    = 0.60    # m

# Fixed for all runs (per validation spec)
_D_PART_M  = 0.001      # 1 mm particle diameter
_T0_PART_K = 293.15     # 20 degC initial particle temperature
_HUMIDITY  = 0.0        # kg/kg (set to 0 for all runs)
_R_SPRAY   = 6.7e-6     # kg/s  (fixed coating loss rate during spraying)

# Dissolution model constants (from MATLAB ModelDissolution)
_DISS = dict(Volume_disso=1000.0, rho_EC=0.4, Permeability=1.5e-7,
             Mass_sample=1.058, Total_min=240)

# Data paths (notebook lives in notebooks/, data in data/)
_NB_DIR  = os.getcwd()
_ROOT    = os.path.dirname(_NB_DIR)
_DATA    = os.path.join(_ROOT, "data")
_DOE_CSV = os.path.join(_DATA, "DoE_recipe_and_inputs.csv")
_PV_DIR  = os.path.join(_DATA, "Process_variables")
_DSP_DIR = os.path.join(_DATA, "Disso_end_spray")
_DDC_DIR = os.path.join(_DATA, "Disso_discharge")

_DOE = pd.read_csv(_DOE_CSV)
_DOE["Run"] = _DOE["Run"].str.strip()
print(f"DoE table loaded: {len(_DOE)} runs")


DoE table loaded: 20 runs


In [23]:
# Stage colours
_C = {"PH": "royalblue", "SP": "darkorange", "DR": "seagreen"}


def _disso_curve(Mc_kg, batch_kg, ssa_cm2g):
    # First-order dissolution: returns (t_min, F_pct, k_1_per_s)
    EC_mass = Mc_kg / batch_kg
    if EC_mass <= 0:
        t_min = np.arange(_DISS["Total_min"] + 1, dtype=float)
        return t_min, np.full_like(t_min, 100.0), np.inf
    S = _DISS["Mass_sample"] * ssa_cm2g
    k = (S * _DISS["Permeability"] * _DISS["rho_EC"] * ssa_cm2g
         / (_DISS["Volume_disso"] * EC_mass))
    t_s = np.arange(0, _DISS["Total_min"] + 1) * 60.0
    F   = 100.0 * (1.0 - np.exp(-k * t_s))
    return t_s / 60.0, F, k


def _load_disso(folder, run_num, tag):
    fname = f"Bosch_DoE_(Ded_2018)_-_Run{run_num:02d}_disso_{tag}.csv"
    df    = pd.read_csv(os.path.join(folder, fname))
    return df["Time"].values.astype(float), df["Average_Disso"].values.astype(float)


def _shade(ax, ph_end, sp_end, t_end):
    ax.axvspan(0,      ph_end, alpha=0.07, color=_C["PH"])
    ax.axvspan(ph_end, sp_end, alpha=0.07, color=_C["SP"])
    ax.axvspan(sp_end, t_end,  alpha=0.07, color=_C["DR"])
    ax.axvline(ph_end, color="grey", lw=0.7, ls="--", alpha=0.5)
    ax.axvline(sp_end, color="grey", lw=0.7, ls="--", alpha=0.5)
    ax.set_xlabel("Time (min)")


def _stage_txt(ax, ph_end, sp_end, t_end):
    lo, hi = ax.get_ylim()
    yp = lo + (hi - lo) * 0.97
    for pos, lbl, c in [
        ((0 + ph_end) / 2,    "PH", _C["PH"]),
        ((ph_end + sp_end) / 2, "SP", _C["SP"]),
        ((sp_end + t_end) / 2,  "DR", _C["DR"]),
    ]:
        ax.text(pos, yp, lbl, ha="center", va="top", fontsize=8,
                color=c, fontweight="bold")

print("Helpers defined.")


Helpers defined.


In [ ]:
def _run06(run_num):
    run_id = f"Run_{run_num:02d}"
    row    = _DOE[_DOE["Run"] == run_id].iloc[0]

    # Parameters from DoE CSV
    batch_kg    = float(row["Batch_size"])
    ssa_cm2g    = float(row["SSA"])
    dmc_pct     = float(row["Coating_Solution_Concentration"])  # e.g. 1.5 %
    dmc_frac    = dmc_pct / 100.0
    qty_sol_kg  = float(row["Qty_solution"])
    coat_lvl    = float(row["Coating_level"])  # coded DoE factor (-1, 0.52, 1)
    h_coded     = float(row["Inlet_air_humidity"])  # coded DoE factor (0, 0.5, 1)

    ph_T_K   = float(row["ph_inlet_T_C"])    + 273.15
    ph_m_kgs = float(row["ph_airflow_m3h"])  / 3600.0 * _RHO_AIR
    ph_dur_s = float(row["ph_duration_min"]) * 60.0

    sp_T_K        = float(row["sp_inlet_T_C"])        + 273.15
    sp_m_kgs      = float(row["sp_airflow_m3h"])       / 3600.0 * _RHO_AIR
    sp_rate_g_min = float(row["sp_spray_rate_g_min"])
    sp_rate_kgs   = sp_rate_g_min / 60_000.0
    sp_dur_s      = qty_sol_kg / sp_rate_kgs

    dr_T_K     = float(row["dr_inlet_T_C"])    + 273.15
    dr_m_kgs   = float(row["dr_airflow_m3h"])  / 3600.0 * _RHO_AIR
    dr_dur_s   = float(row["dr_duration_min"]) * 60.0
    dr_dur_min = dr_dur_s / 60.0

    # r_drying from empirical correlation.
    # First arg: spray rate during spraying [g/min] (DoE factor the correlation
    # was fitted against).  Last arg: coded humidity DoE factor (0, 0.5, 1).
    r_dry = calc_r_drying(sp_rate_g_min, coat_lvl, batch_kg, dr_dur_min, dmc_pct, h_coded)

    # ProcessParameters for each stage
    _PHYS = dict(diameter_eq=_D_PART_M, ssa_cm2_g=ssa_cm2g,
                 particle_density=_RHO_PART, cp_particle=_CP_PART,
                 diameter_bed=_D_BED, rho_air=_RHO_AIR, cp_air=_CP_AIR,
                 batch_size=batch_kg)

    p_ph = ProcessParameters(**_PHYS,
        air_flow_rates=(ph_m_kgs, ph_m_kgs, ph_m_kgs),
        air_temperatures=(ph_T_K, ph_T_K, ph_T_K),
        air_inlet_moisture=(_HUMIDITY, _HUMIDITY, _HUMIDITY))

    p_sp = ProcessParameters(**_PHYS,
        air_flow_rates=(sp_m_kgs, sp_m_kgs, sp_m_kgs),
        air_temperatures=(sp_T_K, sp_T_K, sp_T_K),
        air_inlet_moisture=(_HUMIDITY, _HUMIDITY, _HUMIDITY),
        spray_rate=sp_rate_kgs, dry_matter_conc=dmc_frac, r_spraying=_R_SPRAY)

    p_dr = ProcessParameters(**_PHYS,
        air_flow_rates=(dr_m_kgs, dr_m_kgs, dr_m_kgs),
        air_temperatures=(dr_T_K, dr_T_K, dr_T_K),
        air_inlet_moisture=(_HUMIDITY, _HUMIDITY, _HUMIDITY),
        r_drying=r_dry)

    # Run simulations
    res_ph = run_preheating(p_ph, duration=ph_dur_s, T_particle_init=_T0_PART_K)
    res_sp = run_spraying(p_sp, duration=sp_dur_s,
                          T_particle_init=res_ph.T_particle[-1])
    res_dr = run_drying(p_dr, duration=dr_dur_s,
                        Y_particle_init=res_sp.Y_particle[-1],
                        Y_gas_init=res_sp.Y_gas[-1],
                        M_coating_init=res_sp.M_coating[-1],
                        T_particle_init=res_sp.T_particle[-1])

    # Continuous time axes (minutes)
    t_ph  = res_ph.t / 60.0
    t_sp  = (res_sp.t + res_ph.t[-1]) / 60.0
    t_dr  = (res_dr.t + res_ph.t[-1] + res_sp.t[-1]) / 60.0
    t_all = np.concatenate([t_ph, t_sp, t_dr])
    ph_end, sp_end, t_end = t_ph[-1], t_sp[-1], t_dr[-1]

    # Inlet setpoint step
    ph_T_C = float(row["ph_inlet_T_C"])
    sp_T_C = float(row["sp_inlet_T_C"])
    dr_T_C = float(row["dr_inlet_T_C"])
    t_step = [0, ph_end, ph_end, sp_end, sp_end, t_end]
    T_step = [ph_T_C, ph_T_C, sp_T_C, sp_T_C, dr_T_C, dr_T_C]

    # Model signals
    T_prod = np.concatenate([res_ph.T_particle - 273.15,
                              res_sp.T_particle - 273.15,
                              res_dr.T_particle - 273.15])
    T_gas  = np.concatenate([res_ph.T_gas  - 273.15,
                              res_sp.T_gas  - 273.15,
                              res_dr.T_gas  - 273.15])
    Y_part = np.concatenate([np.zeros_like(t_ph),
                              res_sp.Y_particle * 100,
                              res_dr.Y_particle * 100])
    Y_gas  = np.concatenate([np.zeros_like(t_ph),
                              res_sp.Y_gas * 100,
                              res_dr.Y_gas * 100])
    WG     = np.concatenate([np.zeros_like(t_ph),
                              res_sp.M_coating / batch_kg * 100,
                              res_dr.M_coating / batch_kg * 100])

    # ── Observed process variables ────────────────────────────────────────────
    # Start after the last "Charge" row (same logic as extract_doe_inputs.py)
    # and reset time to zero so axes align with the model.
    pv_file = os.path.join(
        _PV_DIR, f"Bosch_DoE_(Ded_2018)_-_Run{run_num:02d}_process_variables.csv")
    pv = pd.read_csv(pv_file, low_memory=False)
    pv.columns = pv.columns.str.strip()

    phase_raw = pv["ProcessPhase"].str.strip()
    is_charge = phase_raw.str.startswith("Charge")
    if is_charge.any():
        last_charge = int(len(is_charge) - 1 - np.argmax(is_charge.values[::-1]))
        ph_start_idx = last_charge + 1
    else:
        ph_start_idx = 0

    pv_trim    = pv.iloc[ph_start_idx:].copy()
    t0_obs     = float(pv_trim["Time"].iloc[0])
    obs_t      = pv_trim["Time"].values.astype(float) - t0_obs
    obs_T_prod = pv_trim["Product_temp"].values.astype(float)
    obs_T_out  = pv_trim["Outlet_air_temp"].values.astype(float)

    # Observed dissolution (average of 3 aliquots)
    obs_t_sp, obs_F_sp = _load_disso(_DSP_DIR, run_num, "end_spray")
    obs_t_dc, obs_F_dc = _load_disso(_DDC_DIR, run_num, "discharge")

    # Model dissolution curves
    t_d_sp, F_sp, k_sp = _disso_curve(res_sp.M_coating[-1], batch_kg, ssa_cm2g)
    t_d_dc, F_dc, k_dc = _disso_curve(res_dr.M_coating[-1], batch_kg, ssa_cm2g)
    wg_sp = res_sp.M_coating[-1] / batch_kg * 100
    wg_dc = res_dr.M_coating[-1] / batch_kg * 100

    # =========================================================================
    # Figure 1 — Process timeline (2 x 2)
    # =========================================================================
    fig1, axes = plt.subplots(2, 2, figsize=(12, 6))
    fig1.suptitle(
        f"{run_id}  |  batch {batch_kg} kg  |  SSA {ssa_cm2g:.1f} cm2/g  "
        f"|  DMC {dmc_pct:.1f}%  |  hum {h_coded:.1f}  |  r_dry = {r_dry:.2e} kg/s",
        fontsize=11, fontweight="bold")

    # Temperature
    ax = axes[0, 0]
    _shade(ax, ph_end, sp_end, t_end)
    ax.step(t_step, T_step, color="tomato", lw=1.5, ls="--",
            where="post", label="T inlet (setpoint)")
    ax.plot(t_all, T_gas,  color="tomato",    lw=1.5, alpha=0.8, label="T outlet (model)")
    ax.plot(t_all, T_prod, color="steelblue", lw=2.0,            label="T product (model)")
    ax.plot(obs_t, obs_T_out,  marker="+", ls="none", color="tomato",
            ms=5, mew=1.4, alpha=0.5, label="T outlet (obs)")
    ax.plot(obs_t, obs_T_prod, marker="+", ls="none", color="steelblue",
            ms=5, mew=1.4, alpha=0.5, label="T product (obs)")
    ax.set_ylabel("Temperature (degC)"); ax.set_title("Temperature profiles")
    ax.legend(fontsize=7, loc="lower right"); ax.grid(True, alpha=0.3)
    _stage_txt(ax, ph_end, sp_end, t_end)

    # Particle acetone
    ax = axes[0, 1]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, Y_part, color="darkorange", lw=2.0)
    ax.set_ylabel("Acetone on particles (wt %)"); ax.set_title("Particle acetone (model)")
    ax.grid(True, alpha=0.3); _stage_txt(ax, ph_end, sp_end, t_end)

    # Gas acetone
    ax = axes[1, 0]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, Y_gas, color="cadetblue", lw=2.0)
    ax.set_ylabel("Acetone in gas (wt %)"); ax.set_title("Gas-phase acetone (model)")
    ax.grid(True, alpha=0.3); _stage_txt(ax, ph_end, sp_end, t_end)

    # Coating WG
    ax = axes[1, 1]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, WG, color="mediumpurple", lw=2.0)
    ax.set_ylabel("Coating weight gain (%)"); ax.set_title("Coating WG (model)")
    ax.grid(True, alpha=0.3); _stage_txt(ax, ph_end, sp_end, t_end)

    fig1.tight_layout()
    display(fig1); plt.close(fig1)

    # =========================================================================
    # Figure 2 — Dissolution comparison
    # =========================================================================
    fig2, ax = plt.subplots(figsize=(8, 5))
    fig2.suptitle(
        f"{run_id} — Dissolution  |  "
        f"WG end-spray {wg_sp:.2f}%  |  WG discharge {wg_dc:.2f}%",
        fontsize=11, fontweight="bold")

    ax.plot(t_d_sp, F_sp, color=_C["SP"], lw=2.0,
            label=f"End-spray model  (k={k_sp:.3e} s-1, WG={wg_sp:.2f}%)")
    ax.plot(obs_t_sp, obs_F_sp, marker="+", ls="none", color=_C["SP"],
            ms=10, mew=2.0, label="End-spray observed (avg)")

    ax.plot(t_d_dc, F_dc, color=_C["DR"], lw=2.0,
            label=f"Discharge model  (k={k_dc:.3e} s-1, WG={wg_dc:.2f}%)")
    ax.plot(obs_t_dc, obs_F_dc, marker="+", ls="none", color=_C["DR"],
            ms=10, mew=2.0, label="Discharge observed (avg)")

    ax.set_xlim(0, _DISS["Total_min"]); ax.set_ylim(0, 105)
    ax.set_xlabel("Dissolution time (min)"); ax.set_ylabel("Drug released (%)")
    ax.set_title("First-order dissolution — model vs observed")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    fig2.tight_layout()
    display(fig2); plt.close(fig2)

    # Summary line
    print(f"Spray dur: {sp_dur_s/60:.1f} min  |  Total: {t_end:.1f} min  "
          f"|  humidity (coded): {h_coded}  |  r_drying: {r_dry:.3e} kg/s")
    print(f"WG end-spray: {wg_sp:.3f}%  |  WG discharge: {wg_dc:.3f}%")

print("Simulation function defined.")


In [25]:
_w_run = widgets.IntSlider(
    value=1, min=1, max=20, step=1,
    description="Run number:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px"),
    continuous_update=False,
)

_out = widgets.interactive_output(_run06, {"run_num": _w_run})

clear_output(wait=True)
display(
    widgets.HTML(
        '<b style="background:#2E4057;color:white;padding:4px 12px;'
        'border-radius:4px;display:inline-block;margin:6px 0">'
        'DoE Run Selector</b>'),
    _w_run,
    _out,
)


HTML(value='<b style="background:#2E4057;color:white;padding:4px 12px;border-radius:4px;display:inline-block;m…

IntSlider(value=1, continuous_update=False, description='Run number:', layout=Layout(width='400px'), max=20, m…

Output()